In [1]:
import os
import pickle
import pandas as pd
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit

df_file = "canbind_combined_20260806"

# Load stata file and enumerate data

In [2]:
df = pd.read_stata(f"/data/Clinical_vars/{df_file}.dta")
df = df[[
    "study", "SubjectID", "treatment", "age", "sex", "stratum", "week", "blmd", "blqids", "blbmi",
    "madrs", "mdperchange", "mdresp", "mdremit", "qidsrtot", "treatmentstartdate", "madrsdate", "canbindID", "mdvalid"
]]
df = df.dropna(subset=["madrs"])

In [3]:
df

,study,SubjectID,treatment,age,sex,stratum,week,blmd,blqids,blbmi,madrs,mdperchange,mdresp,mdremit,qidsrtot,treatmentstartdate,madrsdate,canbindID,mdvalid
0,cbn1,CBN01_CAM_0005,esc,24.0,F,,0.0,33.000,17.0,NaN,33.000,0.000000,0.0,0.0,17.0,NaT,NaT,,10.0
1,cbn1,CBN01_CAM_0005,esc,24.0,F,,2.0,33.000,17.0,NaN,27.000,-18.181818,0.0,0.0,15.0,NaT,NaT,,10.0
2,cbn1,CBN01_CAM_0005,esc,24.0,F,,4.0,33.000,17.0,NaN,27.000,-18.181818,0.0,0.0,11.0,NaT,NaT,,10.0
3,cbn1,CBN01_CAM_0005,esc,24.0,F,,6.0,33.000,17.0,NaN,19.000,-42.424244,0.0,0.0,NaN,NaT,NaT,,10.0
4,cbn1,CBN01_CAM_0005,esc,24.0,F,,8.0,33.000,17.0,NaN,18.000,-45.454544,0.0,0.0,8.0,NaT,NaT,,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4310,tide,TDE01-MCG-0015,cbt,21.0,F,adu,4.0,39.000,18.0,20.622835,26.000,-33.333332,0.0,0.0,12.0,NaT,2026-07-07,,10.0
4311,tide,TDE01-MCG-0015,cbt,21.0,F,adu,16.0,39.000,18.0,20.622835,34.000,-12.820513,0.0,0.0,15.0,NaT,2026-07-09,,10.0
4312,tide,TDE01-MCG-0017,fluox,24.0,F,adu,0.0,28.806,18.0,24.973988,28.806,0.000000,0.0,0.0,18.0,2026-07-03,NaT,,0.0
4313,tide,TDE01-MCG-0017,fluox,24.0,F,adu,1.0,26.000,18.0,24.973988,8.000,-69.230766,1.0,1.0,6.0,2026-07-03,2026-07-14,,10.0


In [4]:
audio_files = sorted(list(glob("/data/Audio_sent_for_transcription/*.wav")))
len(audio_files)

1840

In [5]:
audio_records = []
for fname in tqdm(audio_files):
    base_fname = os.path.splitext(os.path.basename(fname))[0]
    fname_parts = base_fname.replace("-", " ").replace("_", " ").split()
    if base_fname[:5] == "CBN17":
        # optimd
        study = "optimd"
        subject_id = base_fname[:14]
        week = fname_parts[3]
        if len(week) > 2:
            week = fname_parts[4]
    elif base_fname[:5] == "TDE01":
        # tide
        study = "tide"
        subject_id = base_fname[:14].replace("_", "-")
        week = fname_parts[3]
    elif base_fname[:2] == "MD":
        # possible cbtadm
        study = "cbtadm"
        subject_id = base_fname[:8]
        week = fname_parts[1][:2]
        if not week.isnumeric():
            continue
    else:
        continue
    trans_fname = f"/data/CrisperWhisper_output/raw_transcripts/{base_fname}.pkl"
    if not os.path.exists(trans_fname):
        continue
    trans_word_fname = f"/data/CrisperWhisper_output/word_level_timestamps/{base_fname}.pkl"
    if not os.path.exists(trans_word_fname):
        tmp = pickle.load(open(trans_fname, "rb"))
        new_trans = [(x.word, x.start, x.end) for x in tmp.words]
        pickle.dump(new_trans, open(trans_word_fname, "wb"))
    diar_fname = f"/data/CrisperWhisper_output/raw_diarization/{base_fname}.pkl"
    if not os.path.exists(diar_fname):
        continue

    audio_records.append({"audio_path": fname, "word_timestamps_path": trans_word_fname, "diarization_path": diar_fname, "study": study, "SubjectID": subject_id, "week": float(week)})
audio_df = pd.DataFrame(audio_records)
audio_df = audio_df.drop_duplicates(subset=["study", "SubjectID", "week"], keep="first")

100%|██████████| 1840/1840 [00:00<00:00, 294809.36it/s]


In [6]:
audio_df

,audio_path,word_timestamps_path,diarization_path,study,SubjectID,week
0,/data/Audio_sent_for_transcription/CBN17_CAM_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...,optimd,CBN17_CAM_0001,0.0
1,/data/Audio_sent_for_transcription/CBN17_CAM_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...,optimd,CBN17_CAM_0001,8.0
2,/data/Audio_sent_for_transcription/CBN17_CAM_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...,optimd,CBN17_CAM_0002,0.0
3,/data/Audio_sent_for_transcription/CBN17_CAM_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...,optimd,CBN17_CAM_0002,8.0
4,/data/Audio_sent_for_transcription/CBN17_CAM_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...,optimd,CBN17_CAM_0005,0.0
...,...,...,...,...,...,...
863,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...,tide,TDE01-MCG-0010,0.0
864,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...,tide,TDE01-MCG-0010,16.0
866,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...,tide,TDE01-MCG-0011,0.0
867,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...,tide,TDE01-MCG-0013,0.0


In [7]:
df = df.merge(audio_df, on=['study', 'SubjectID', 'week'], how='left')
df = df.dropna(subset=["audio_path", "madrs"])
df

,study,SubjectID,treatment,age,sex,stratum,week,blmd,blqids,blbmi,...,mdresp,mdremit,qidsrtot,treatmentstartdate,madrsdate,canbindID,mdvalid,audio_path,word_timestamps_path,diarization_path
2060,cbtadm,MD000117,cbt,46.0,F,mdd,0.0,24.0,12.0,30.033756,...,0.0,0.0,12.0,2018-01-31,2018-01-26,CBN06_DAL_0018,10.0,/data/Audio_sent_for_transcription/MD000117-00...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/MD...
2106,cbtadm,MD000154,esc,25.0,M,pdd,0.0,29.0,NaN,27.688745,...,0.0,0.0,NaN,2018-10-18,2018-10-10,CBN06_DAL_0023,10.0,/data/Audio_sent_for_transcription/MD000154-00...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/MD...
2115,cbtadm,MD000161,cbt,36.0,F,pdd,0.0,29.0,21.0,31.177980,...,0.0,0.0,21.0,2018-11-05,2018-10-31,CBN06_DAL_0025,10.0,/data/Audio_sent_for_transcription/MD000161-00...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/MD...
2148,cbtadm,MD000281,cbt,26.0,F,mdd,0.0,26.0,17.0,28.688545,...,0.0,0.0,17.0,2019-02-15,2019-02-06,CBN06_DAL_0029,10.0,/data/Audio_sent_for_transcription/MD000281-00...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/MD...
2153,cbtadm,MD000282,cbt,51.0,M,pdd,0.0,24.0,8.0,30.064161,...,0.0,0.0,8.0,2019-02-06,2019-02-12,CBN06_DAL_0030,10.0,/data/Audio_sent_for_transcription/MD000282-00...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/MD...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4223,tide,TDE01-MCG-0010,cbt,23.0,F,adu,0.0,37.0,19.0,24.740482,...,0.0,0.0,19.0,2026-02-18,2026-01-28,,10.0,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...
4231,tide,TDE01-MCG-0010,cbt,23.0,F,adu,16.0,37.0,19.0,24.740482,...,0.0,0.0,19.0,2026-02-18,2026-04-29,,10.0,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...
4232,tide,TDE01-MCG-0011,fluox,19.0,F,adu,0.0,35.0,18.0,18.833401,...,0.0,0.0,18.0,2026-02-18,2026-02-16,,10.0,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...
4241,tide,TDE01-MCG-0013,cbt,17.0,M,ado,0.0,38.0,24.0,30.960333,...,0.0,0.0,24.0,2026-03-30,2026-03-23,,10.0,/data/Audio_sent_for_transcription/TDE01_MCG_0...,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...


In [8]:
df["regression_label"] = df["madrs"]
df["subject_id"] = df["SubjectID"]
df["class_label"] = pd.cut(
    df["regression_label"],
    bins=[-float("inf"), 4, 11, 22, float("inf")],
    labels=[0, 1, 2, 3],
).astype("Int64")

In [9]:
df = df[["audio_path", "word_timestamps_path", "diarization_path", "class_label", "regression_label", "subject_id"]]

In [66]:
manifest_path = f"/data/Clinical_vars/{df_file}_manifest.csv"
df.to_csv(manifest_path, index=False)

# Train/Validation

In [ ]:
df = pd.read_csv(manifest_path, dtype={"subject_id": str})

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=40,
)

train_idx, validation_idx = next(
    splitter.split(df, y=df["class_label"], groups=df["subject_id"])
)

train_df = df.iloc[train_idx].reset_index(drop=True)
validation_df = df.iloc[validation_idx].reset_index(drop=True)

assert set(train_df["subject_id"]).isdisjoint(validation_df["subject_id"])

train_df.to_csv(
    "/data/Clinical_vars/canbind_combined_20260806_train.csv",
    index=False,
)
validation_df.to_csv(
    "/data/Clinical_vars/canbind_combined_20260806_validation.csv",
    index=False,
)

# Testing

In [68]:
print(train_df["class_label"].value_counts(normalize=True).sort_index())
print(validation_df["class_label"].value_counts(normalize=True).sort_index())

class_label
0    0.082430
1    0.140998
2    0.203905
3    0.572668
Name: proportion, dtype: float64
class_label
0    0.059322
1    0.135593
2    0.254237
3    0.550847
Name: proportion, dtype: float64


In [55]:
test = pickle.load(open(df["diarization_path"].iloc[0], "rb"))

In [56]:
test

[(24.97221875, 29.730968750000002, 'SPEAKER_00'),
 (29.984093750000003, 30.034718750000003, 'SPEAKER_00'),
 (30.051593750000002, 33.00471875, 'SPEAKER_00'),
 (33.22409375, 35.89034375, 'SPEAKER_00'),
 (36.16034375, 41.391593750000006, 'SPEAKER_00'),
 (41.86409375, 43.85534375, 'SPEAKER_00'),
 (50.70659375, 54.82409375, 'SPEAKER_00'),
 (55.27971875, 57.87846875, 'SPEAKER_00'),
 (59.66721875, 66.21471875, 'SPEAKER_01'),
 (66.45096875, 71.69909375, 'SPEAKER_01'),
 (72.05346875000001, 78.75284375000001, 'SPEAKER_01'),
 (79.20846875000001, 96.84284375, 'SPEAKER_01'),
 (97.21409375, 103.20471875000001, 'SPEAKER_01'),
 (103.62659375000001, 127.97721875000002, 'SPEAKER_01'),
 (129.22596875000002, 137.00534375, 'SPEAKER_01'),
 (137.68034375000002, 145.74659375000002, 'SPEAKER_01'),
 (146.21909375, 165.97971875000002, 'SPEAKER_01'),
 (166.67159375, 177.94409375, 'SPEAKER_01'),
 (178.45034375, 179.17596875, 'SPEAKER_01'),
 (179.83409375000002, 183.05721875, 'SPEAKER_01'),
 (183.39471875, 185.5378

In [ ]:
audit_df = pd.read_csv("outputs/hpo_text_regression/screening/trial-0036/recording_loss_audit.csv")
audit_df

,loss_rank,recording_index,audio_path,subject_id,speaker,class_truth,regression_truth,regression_prediction,regression_truth_standardized,regression_prediction_standardized,...,class_truth_probability,classification_nll,classification_loss,regression_error,regression_absolute_error,regression_squared_error_original_scale,regression_loss,combined_loss,word_timestamps_path,diarization_path
0,1,115,/data/Audio_sent_for_transcription/TDE01_MCG_0...,TDE01-MCG-0007,SPEAKER_00,3,44.0,25.029113,1.992229,0.262536,...,0.259867,1.347587,1.347587,-18.970887,18.970887,359.894569,2.991838,2.991838,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/TD...
1,2,0,/data/Audio_sent_for_transcription/MD000288-00...,MD000288,SPEAKER_00,3,41.0,22.612134,1.718701,0.042165,...,0.247670,1.395660,1.395660,-18.387866,18.387866,338.113620,2.810771,2.810771,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/MD...
2,3,78,/data/Audio_sent_for_transcription/CBN17_UBC_0...,CBN17_UBC_0005,SPEAKER_01,0,3.0,20.977765,-1.745994,-0.106850,...,0.286819,1.248904,1.248904,17.977765,17.977765,323.200030,2.686793,2.686793,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...
3,4,53,/data/Audio_sent_for_transcription/CBN17_MCU_0...,CBN17_MCU_0051,SPEAKER_01,3,29.0,11.177291,0.624586,-1.000420,...,0.244002,1.410579,1.410579,-17.822709,17.822709,317.648945,2.640646,2.640646,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...
4,5,100,/data/Audio_sent_for_transcription/CBN17_UHN_0...,CBN17_UHN_0016,SPEAKER_00,0,0.0,17.733273,-2.019523,-0.402671,...,0.269908,1.309672,1.309672,17.733273,17.733273,314.468985,2.614211,2.614211,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,113,68,/data/Audio_sent_for_transcription/CBN17_ROM_0...,CBN17_ROM_0006,SPEAKER_01,3,32.0,31.622309,0.898115,0.863679,...,0.211855,1.551851,1.551851,-0.377691,0.377691,0.142650,0.001186,0.001186,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...
113,114,94,/data/Audio_sent_for_transcription/CBN17_UCA_0...,CBN17_UCA_0018,SPEAKER_01,1,11.0,10.771364,-1.016585,-1.037431,...,0.201482,1.602054,1.602054,-0.228636,0.228636,0.052274,0.000435,0.000435,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...
114,115,49,/data/Audio_sent_for_transcription/CBN17_MCU_0...,CBN17_MCU_0009,SPEAKER_01,2,16.0,16.110036,-0.560704,-0.550671,...,0.238497,1.433398,1.433398,0.110036,0.110036,0.012108,0.000101,0.000101,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...
115,116,89,/data/Audio_sent_for_transcription/CBN17_UCA_0...,CBN17_UCA_0010,SPEAKER_01,3,28.0,28.047006,0.533410,0.537696,...,0.213427,1.544459,1.544459,0.047006,0.047006,0.002210,0.000018,0.000018,/data/CrisperWhisper_output/word_level_timesta...,/data/CrisperWhisper_output/raw_diarization/CB...


In [13]:
test = pickle.load(open(audit_df["word_timestamps_path"].iloc[0], "rb"))

In [14]:
test

[('We', 26.0, 26.02),
 ('are', 26.02, 26.18),
 ('looking', 26.18, 26.48),
 ('at', 26.48, 26.7),
 ('speech', 26.7, 27.08),
 ('behavior', 27.08, 27.56),
 ('as', 27.56, 27.72),
 ('part', 27.72, 27.98),
 ('of', 27.98, 28.08),
 ("today's", 28.08, 28.48),
 ('visit.', 28.48, 28.82),
 ('To', 29.44, 29.52),
 ('do', 29.52, 29.6),
 ('that,', 29.6, 29.86),
 ("I'm", 29.86, 29.98),
 ('going', 29.98, 30.22),
 ('to', 30.22, 30.3),
 ('have', 30.3, 30.54),
 ('you', 30.54, 30.58),
 ('talk', 31.04, 31.44),
 ('about', 31.44, 31.7),
 ('yourself.', 31.7, 32.04),
 ('I', 32.66, 32.7),
 ('would', 32.7, 32.78),
 ('like', 32.78, 32.96),
 ('to', 32.96, 33.04),
 ('get', 33.04, 33.16),
 ('a', 33.16, 33.22),
 ('sense', 33.22, 33.54),
 ('of', 33.54, 33.68),
 ('who', 33.68, 33.74),
 ('you', 33.74, 33.9),
 ('are', 33.9, 34.32),
 ('in', 34.32, 34.5),
 ('your', 34.5, 34.6),
 ('own', 34.6, 34.78),
 ('words.', 34.78, 35.1),
 ('I', 35.9, 35.96),
 ('will', 35.96, 36.08),
 ('give', 36.08, 36.3),
 ('you', 36.3, 36.38),
 ('a', 3

In [15]:
test2 = pickle.load(open(audit_df["diarization_path"].iloc[0], "rb"))

In [16]:
test2

[(25.984718750000003, 28.85346875, 'SPEAKER_01'),
 (29.39346875, 30.625343750000003, 'SPEAKER_01'),
 (30.92909375, 32.14409375, 'SPEAKER_01'),
 (32.58284375, 35.400968750000004, 'SPEAKER_01'),
 (35.83971875, 38.84346875, 'SPEAKER_01'),
 (39.299093750000004, 41.94846875, 'SPEAKER_01'),
 (42.235343750000006, 44.580968750000004, 'SPEAKER_01'),
 (45.22221875, 46.791593750000004, 'SPEAKER_01'),
 (47.26409375, 47.60159375, 'SPEAKER_00'),
 (49.727843750000005, 52.782218750000006, 'SPEAKER_01'),
 (53.119718750000004, 54.92534375, 'SPEAKER_01'),
 (55.532843750000005, 56.76471875, 'SPEAKER_01'),
 (57.102218750000006, 63.227843750000005, 'SPEAKER_01'),
 (63.61596875, 64.74659375, 'SPEAKER_01'),
 (65.16846875, 66.60284375, 'SPEAKER_01'),
 (66.99096875000001, 67.34534375, 'SPEAKER_01'),
 (71.73284375, 72.34034375, 'SPEAKER_00'),
 (72.79596875, 80.72721875, 'SPEAKER_00'),
 (81.35159375, 89.13096875000001, 'SPEAKER_00'),
 (89.45159375, 99.93096875, 'SPEAKER_00'),
 (100.58909375, 103.67721875000001, '